In [2]:
import numpy as np
import pandas as pd
from datetime import datetime

In [3]:
df= pd.read_csv("../augmented/HRDataset_v14_Enriched.csv")


In [6]:
def clean_hr_data(df):
    # Création d'une copie pour ne pas modifier l'original
    df_final = df.copy()

    # --- ÉTAPE 1 : Suppression des colonnes inutiles ou trop spécifiques ---
    # On retire les identifiants uniques, les noms et les colonnes redondantes
    # On retire aussi 'Exit_Interview_Feedback' (colonne 36) comme demandé
    cols_to_drop = [
        'Employee_Name', 'EmpID', 'MaritalStatusID', 'GenderID', 'DeptID', 
        'PerfScoreID', 'PositionID', 'ManagerID', 'Zip', 'ManagerName',
        'Exit_Interview_Feedback', 'TermReason'
    ]
    df_final = df_final.drop(columns=cols_to_drop)

    # --- ÉTAPE 2 : Traitement des Dates ---
    # Conversion en datetime
    date_cols = ['DOB', 'DateofHire', 'DateofTermination', 'LastPerformanceReview_Date']
    for col in date_cols:
        df_final[col] = pd.to_datetime(df_final[col], errors='coerce')

    # Calcul de l'âge (basé sur aujourd'hui)
    current_year = datetime.now().year
    df_final['Age'] = current_year - df_final['DOB'].dt.year
    
    # Calcul de l'ancienneté (Tenure) en années
    # Si DateofTermination est vide, on utilise la date d'aujourd'hui pour le calcul
    end_date = df_final['DateofTermination'].fillna(pd.to_datetime(datetime.now().date()))
    df_final['Tenure_Years'] = (end_date - df_final['DateofHire']).dt.days / 365.25

    # Temps depuis la dernière revue de performance
    df_final['Years_Since_Last_Review'] = (pd.to_datetime(datetime.now().date()) - df_final['LastPerformanceReview_Date']).dt.days / 365.25

    # On supprime les colonnes de dates originales après transformation
    df_final = df_final.drop(columns=date_cols)

    # --- ÉTAPE 3 : Encodage des variables binaires (Oui/Non, Homme/Femme) ---
    df_final['Sex'] = df_final['Sex'].map({'M ': 1, 'Male': 1, 'F': 0, 'Female': 0})
    df_final['HispanicLatino'] = df_final['HispanicLatino'].map({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0})
    df_final['Internal_Transfer_Request'] = df_final['Internal_Transfer_Request'].map({'Yes': 1, 'No': 0}).fillna(0)

    # --- ÉTAPE 4 : Encodage des variables catégorielles (One-Hot Encoding) ---
    # Pour les colonnes avec plusieurs catégories (Position, Department, etc.)
    categorical_cols = [
        'MaritalDesc', 'CitizenDesc', 'RaceDesc', 'EmploymentStatus', 
        'Department', 'Position', 'RecruitmentSource', 'PerformanceScore'
    ]
    
    # On utilise get_dummies pour transformer ces colonnes en colonnes numériques (0 ou 1)
    df_final = pd.get_dummies(df_final, columns=categorical_cols, drop_first=True)

    # --- ÉTAPE 5 : Gestion des valeurs manquantes restantes ---
    # On remplit les éventuels NA numériques par la médiane
    df_final = df_final.fillna(df_final.median(numeric_only=True))

    return df_final

# Application du nettoyage
df_final = clean_hr_data(df)

# Affichage du résultat
print(f"Ancien nombre de colonnes : {df.shape[1]}")
print(f"Nouveau nombre de colonnes : {df_final.shape[1]}")
print(df_final.head())

Ancien nombre de colonnes : 41
Nouveau nombre de colonnes : 80
   MarriedID  EmpStatusID  FromDiversityJobFairID  Salary  Termd State  Sex  \
0          0            1                       0   62506      0    MA    1   
1          1            5                       0  104437      1    MA    1   
2          1            5                       0   64955      1    MA    0   
3          1            1                       0   64991      0    MA    0   
4          0            5                       0   50825      1    MA    0   

   HispanicLatino  EngagementSurvey  EmpSatisfaction  ...  \
0               0              4.60                5  ...   
1               0              4.96                3  ...   
2               0              3.02                3  ...   
3               0              4.84                5  ...   
4               0              5.00                4  ...   

   RecruitmentSource_Employee Referral  RecruitmentSource_Google Search  \
0                   

C:\Users\malor\AppData\Local\Temp\ipykernel_41080\2730638754.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_final[col] = pd.to_datetime(df_final[col], errors='coerce')


In [7]:
df_final.to_csv("HRDataset_cleaned.csv", index=False)